# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import numpy as np

class WarehouseMDP:
    def __init__(self, living_reward=-1.0, gamma=0.9, p_slip=0.6):
        self.height = 5
        self.width = 6

        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        self.terminal_states = {
            (0, 5): 10.0,   # entrega
            (2, 2): 2.0,    # carga
            (3, 5): -10.0,  # peligro mortal
        }

        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = living_reward
        self.gamma = gamma

        # probabilidades (intencion, desvio izq, desvio der)
        self.p_normal = (0.9, 0.05, 0.05)
        p_side = (1 - p_slip) / 2
        self.p_slippery = (p_slip, p_side, p_side)

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state
        if r < 0 or r >= self.height or c < 0 or c >= self.width:
            return False
        return state not in self.walls

    def states(self):
        S = []
        for r in range(self.height):
            for c in range(self.width):
                if self.is_valid_state((r, c)):
                    S.append((r, c))
        return S

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        dr, dc = action
        left = (-dc, dr)
        right = (dc, -dr)

        if state in self.slippery_states:
            p_int, p_left, p_right = self.p_slippery
        else:
            p_int, p_left, p_right = self.p_normal

        transitions = []
        for move, p in [(action, p_int), (left, p_left), (right, p_right)]:
            next_state = (state[0] + move[0], state[1] + move[1])
            if not self.is_valid_state(next_state):
                next_state = state
            transitions.append((next_state, p))
        return transitions


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    total = 0.0
    for next_state, prob in grid.get_transition_probs(state, action):
        total += prob * V[next_state]
    return total


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}

    for it in range(max_iter):
        V_new = V.copy()
        delta = 0.0

        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                best = max(expected_next_value(grid, s, a, V) for a in grid.actions)
                V_new[s] = grid.get_reward(s) + grid.gamma * best
            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new
        if delta < threshold:
            break

    return V, it + 1


def extract_policy(grid, V):
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        policy[s] = max(grid.actions, key=lambda a: expected_next_value(grid, s, a, V))
    return policy


## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}

    for it in range(max_iter):
        V_new = V.copy()
        delta = 0.0

        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                V_new[s] = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, policy[s], V)
            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new
        if delta < threshold:
            break

    return V


def policy_improvement(grid, V):
    # es lo mismo que extraer la politica greedy con respecto a V
    return extract_policy(grid, V)


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # politica inicial: todos a la derecha
    policy = {s: (0, 1) for s in grid.states() if not grid.is_terminal(s)}
    history = []

    for i in range(max_iter):
        V = policy_evaluation(grid, policy, threshold)
        new_policy = policy_improvement(grid, V)

        changes = sum(1 for s in policy if policy[s] != new_policy[s])
        history.append(changes)

        policy = new_policy
        if changes == 0:
            break

    return policy, V, history


## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [19, 7, 1, 0]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


### Respuestas

**1.** Desde `START` el robot va a la **estación de carga +2**. Siguiendo las flechas el camino es (0,0) → (0,1) → (0,2) → (1,2) → (2,2). No intenta ir a la entrega +10.

**2.** Porque para llegar al +10 hay que dar un rodeo largo: la estantería en (0,3) tapa el camino directo y la única forma de llegar a (0,4) o (1,5) es pasando por el peligro de (1,4), que quita 3. Si sumo los pasos con costo 1 cada uno, el peligro y el descuento de gamma = 0.9 (el +10 llega muy tarde y se descuenta bastante), sale mejor terminar rápido en el +2 que está a solo 4 pasos.

**3.** Comparé la política con una versión del mapa sin piso resbaloso (poniendo 0.9 en vez de 0.6) y cambian dos estados. En (1,2), que es resbaloso, sin piso resbaloso el robot iría a la derecha hacia el +10, pero como resbala prefiere bajar directo al +2 y asegurar algo. En (3,1) sin piso resbaloso iría hacia arriba, pero con el resbaloso en (2,1) prefiere ir a la derecha para no pasar por esa celda. En el resto la política queda igual.

**4.** El costo por paso hace que quedarse dando vueltas sea malo, entonces obliga al robot a terminar el episodio rápido. Por eso le conviene más una recompensa pequeña cercana que una grande lejana. También explica por qué en las esquinas de abajo los valores son negativos: desde ahí cualquier camino cuesta muchos pasos.

**5.** Porque ahora la probabilidad de desviarse depende del estado donde está el robot: en piso normal va bien el 90 % de las veces y en piso resbaloso solo el 60 %. Entonces T(s,a,s') tiene que mirar si s está en `slippery_states` antes de repartir la probabilidad, no se puede usar una tabla fija de 0.8 y 0.1 como en el GridWorld de clase.

In [7]:
# funcion para ver a donde llega el robot desde START si siempre logra moverse en la direccion elegida
def path_from_start(grid, policy, max_steps=30):
    s = grid.start
    path = [s]
    for _ in range(max_steps):
        if grid.is_terminal(s):
            break
        a = policy[s]
        nxt = (s[0] + a[0], s[1] + a[1])
        if grid.is_valid_state(nxt):
            s = nxt
        path.append(s)
    return path


def resolver(grid, mostrar=True):
    V, n = value_iteration(grid)
    pi = extract_policy(grid, V)
    if mostrar:
        print("Iteraciones:", n)
        print_values(grid, V)
        print()
        print_policy(grid, pi)
        print("\nCamino desde START:", path_from_start(grid, pi))
    return V, pi

print("Caso base")
V_base, pi_base = resolver(WarehouseMDP())

Caso base
Iteraciones: 20
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

Camino desde START: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)]


### Experimento A: menos costo por paso

**Predicción:** como moverse cuesta mucho menos, creo que el robot ya no va a tener afán de terminar en el +2 y va a preferir ir por el +10 aunque tenga que pasar por el peligro de (1,4).

In [8]:
grid_A = WarehouseMDP(living_reward=-0.1)
V_A, pi_A = resolver(grid_A)

Iteraciones: 26
 +1.649 |  +1.992 |  +2.361 |   WALL   |  +8.591 | +10.000
 +1.358 |   WALL   |  +2.797 |  +3.911 |  +4.550 |  +8.591
 +1.259 |  +1.533 |  +2.000 |  +3.306 |   WALL   |  +7.537
 +1.243 |  +1.540 |  +2.040 |  +2.417 |  +2.026 | -10.000
 +0.865 |  -1.794 |   WALL   |  +2.026 |  +1.709 |  +0.874

 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

Camino desde START: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4), (1, 5), (0, 5)]


**Resultado A:** se cumplió la predicción. Desde START ahora el camino es (0,0) → (0,1) → (0,2) → (1,2) → (1,3) → (1,4) → (1,5) → (0,5), o sea que va por la entrega +10. El valor de START pasó de negativo (2.575 negativo) a 1.649 positivo. En (1,2) la acción cambió de bajar a ir a la derecha (también cambiaron (1,0) y (3,2), que ahora apuntan hacia el camino largo). Aunque pasa por el peligro de (1,4), con pasos tan baratos vale la pena.

### Experimento B: piso muy resbaloso

**Predicción:** con 0.40 el piso resbaloso es más peligroso, entonces pienso que el robot va a evitar todavía más esas celdas y que los valores alrededor bajan. La política desde START no debería cambiar porque ya iba al +2.

In [9]:
grid_B = WarehouseMDP(p_slip=0.4)
V_B, pi_B = resolver(grid_B)

print("\nEstados donde cambia la politica frente al caso base:")
print([s for s in pi_base if pi_base[s] != pi_B[s]])

for s in [(0, 0), (1, 2), (2, 1), (3, 3)]:
    print(s, "base:", round(V_base[s], 3), " B:", round(V_B[s], 3))

Iteraciones: 22
 -2.706 |  -1.809 |  -0.797 |   WALL   |  +7.607 | +10.000
 -2.652 |   WALL   |  +0.395 |  +2.104 |  +3.670 |  +7.607
 -1.744 |  -0.670 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.831 |  -0.774 |  +0.534 |  -1.137 |  -2.152 | -10.000
 -2.785 |  -3.929 |   WALL   |  -2.152 |  -2.974 |  -4.041

 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

Camino desde START: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)]

Estados donde cambia la politica frente al caso base:
[]
(0, 0) base: -2.575  B: -2.706
(1, 2) base: 0.56  B: 0.395
(2, 1) base: -0.063  B: -0.67
(3, 3) base: -0.782  B: -1.137


**Resultado B:** la política quedó exactamente igual a la del caso base, no cambió ningún estado. Lo que sí cambió fueron los valores: bajaron sobre todo en las celdas resbalosas, por ejemplo (2,1) bajó como 0.6 y (3,3) como 0.35, y START bajó un poco (de 2.575 negativo a 2.706 negativo). Tiene sentido porque la política ya estaba evitando los riesgos, solo que ahora es más probable desviarse y llegar a un lado peor, entonces el valor esperado baja.

### Experimento C: más paciencia

**Predicción:** con gamma = 0.99 el +10 casi no se descuenta aunque esté lejos, así que creo que el robot sí va a buscar la entrega.

In [10]:
grid_C = WarehouseMDP(gamma=0.99)
V_C, pi_C = resolver(grid_C)

print("\nEstados donde cambia la politica frente al caso base:")
print([s for s in pi_base if pi_base[s] != pi_C[s]])

Iteraciones: 24
 -1.456 |  -0.310 |  +0.809 |   WALL   |  +8.601 | +10.000
 -2.179 |   WALL   |  +2.003 |  +4.118 |  +5.354 |  +8.601
 -1.081 |  +0.119 |  +2.000 |  +2.913 |   WALL   |  +7.395
 -1.606 |  -0.467 |  +0.799 |  +0.817 |  -0.359 | -10.000
 -2.752 |  -3.737 |   WALL   |  -0.359 |  -1.408 |  -2.892

 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

Camino desde START: [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4), (1, 5), (0, 5)]

Estados donde cambia la politica frente al caso base:
[(1, 2)]


**Resultado C:** sí, la política valora más el +10 distante. Desde START el robot ahora hace el mismo camino largo del experimento A y termina en (0,5). El único estado que cambió fue (1,2), que pasó de bajar al +2 a ir a la derecha. Con gamma alto la recompensa de 10 llega casi completa aunque tome 7 pasos, entonces compensa el costo de los pasos y el peligro de (1,4).

### Bonus: valor de living_reward donde cambia la decisión

In [11]:
for lr in np.arange(-0.1, -1.6, -0.1):
    g = WarehouseMDP(living_reward=lr)
    V, pi = resolver(g, mostrar=False)
    print(f"living_reward = {lr:.2f}, termina en {path_from_start(g, pi)[-1]}")

living_reward = -0.10, termina en (0, 5)
living_reward = -0.20, termina en (0, 5)
living_reward = -0.30, termina en (0, 5)
living_reward = -0.40, termina en (0, 5)
living_reward = -0.50, termina en (0, 5)
living_reward = -0.60, termina en (0, 5)
living_reward = -0.70, termina en (0, 5)
living_reward = -0.80, termina en (2, 2)
living_reward = -0.90, termina en (2, 2)
living_reward = -1.00, termina en (2, 2)
living_reward = -1.10, termina en (2, 2)
living_reward = -1.20, termina en (2, 2)
living_reward = -1.30, termina en (2, 2)
living_reward = -1.40, termina en (2, 2)
living_reward = -1.50, termina en (2, 2)


In [12]:
# miro mas fino entre 0.7 y 0.8
for lr in np.arange(-0.70, -0.81, -0.01):
    g = WarehouseMDP(living_reward=lr)
    V, pi = resolver(g, mostrar=False)
    print(f"living_reward = {lr:.2f}, termina en {path_from_start(g, pi)[-1]}  accion en (1,2): {ARROWS[pi[(1, 2)]]}")

living_reward = -0.70, termina en (0, 5)  accion en (1,2): →
living_reward = -0.71, termina en (0, 5)  accion en (1,2): →
living_reward = -0.72, termina en (0, 5)  accion en (1,2): →
living_reward = -0.73, termina en (0, 5)  accion en (1,2): →
living_reward = -0.74, termina en (0, 5)  accion en (1,2): →
living_reward = -0.75, termina en (0, 5)  accion en (1,2): →
living_reward = -0.76, termina en (0, 5)  accion en (1,2): →
living_reward = -0.77, termina en (0, 5)  accion en (1,2): →
living_reward = -0.78, termina en (0, 5)  accion en (1,2): →
living_reward = -0.79, termina en (0, 5)  accion en (1,2): →
living_reward = -0.80, termina en (2, 2)  accion en (1,2): ↓
living_reward = -0.81, termina en (2, 2)  accion en (1,2): ↓


**Respuesta bonus:** el cambio está aproximadamente en **living_reward ≈ 0.8 negativo** (entre 0.79 y 0.80). Con un costo por paso de 0.79 o menos el robot todavía va por la entrega +10, y desde 0.80 en adelante ya prefiere ir a la carga +2. La decisión se define en la celda (1,2), que es donde el robot escoge entre bajar al +2 o seguir a la derecha hacia el +10.